# Capítulo 17: Extender Polars

Como has visto en capítulos anteriores, la API de Polars ya es bastante extensa y cubre una amplia gama de funcionalidades. Sin embargo, puede haber casos en los que desees extender Polars con tu propia funcionalidad personalizada. Esto podría deberse a que tienes un caso de uso específico que no está cubierto por las funciones integradas o porque deseas optimizar el rendimiento de tu código.

En este capítulo, aprenderás a:

- Aplicar una función personalizada de Python a datos de Polars
- Registrar un espacio de nombres en Polars
- Escribir complementos en Rust y ejecutarlos en el motor de Polars para un rendimiento máximo
- Usar crates de Rust en esos complementos

In [1]:
import polars as pl

## Funciones definidas por el usuario (UDF) en Python

Polars ofrece una amplia variedad de expresiones para realizar operaciones sobre tus datos. Sin embargo, 
en ocasiones puedes necesitar una operación que no está cubierta por las expresiones integradas, o que depende de un paquete externo. 
Para estos casos, Polars permite el uso de funciones definidas por el usuario (UDFs).

Los métodos principales para aplicar UDFs en Polars son:

- **Expr.map_elements()**  
    Aplica una función de Python a cada elemento de una Serie.

- **Expr.map_batches()**  
    Aplica una función de Python a una Serie o a una secuencia de Series.

- **Expr.map_groups()**  
    Aplica una función de Python a cada grupo en el contexto de un GroupBy.

- **Expr.pipe()**  
    Aplica una función de Python a una Expresión.

- **df.pipe()**  
    Aplica una función de Python a todo un DataFrame (o LazyFrame).

A continuación, veremos cómo utilizar estos métodos para aplicar funciones personalizadas a tus datos en Polars.

## Aplicando una función a elementos

In [2]:
from textblob import TextBlob

In [3]:
def analyze_sentiment(review):
    return TextBlob(review ).sentiment.polarity

In [4]:
reviews = pl.DataFrame(
    {
        "reviews": [
        "This product is great!",
        "Terrible service.",
        "Okay, but not what I expected.",
        "Excellent! I love it.",
        ]
    }
)

reviews 

reviews
str
"""This product is great!"""
"""Terrible service."""
"""Okay, but not what I expected."""
"""Excellent! I love it."""


In [5]:
reviews.with_columns(
    sentiment_score=pl.col("reviews").map_elements(
        analyze_sentiment, return_dtype=pl.Float64
    )
)

reviews,sentiment_score
str,f64
"""This product is great!""",1.0
"""Terrible service.""",-1.0
"""Okay, but not what I expected.""",0.2
"""Excellent! I love it.""",0.75


## Applying a Function to a Series

El método `Expr.map_batches()` te permite aplicar una función de Python a una Serie o a una secuencia de Series. Esto es útil cuando necesitas información sobre otros elementos de la Serie, o cuando necesitas aplicar una función a varias Series al mismo tiempo.

**Tabla 17-1. Argumentos del método Expr.map_batches()**

| Argumento       | Descripción                                                                                                   |
|-----------------|--------------------------------------------------------------------------------------------------------------|
| function        | Función a aplicar sobre la Serie.                                                                            |
| return_dtype    | Tipo de dato de la Serie que retorna la función.                                                             |
| is_elementwise  | Indica si la función puede aplicarse elemento a elemento. Si es así, puede ejecutarse en el motor streaming. |
| agg_list        | Agrega los valores de la expresión en una lista antes de aplicar la función en un contexto de GroupBy.        |

In [6]:
from scipy.special import softmax
import polars.selectors as cs

ml_dataset = pl.DataFrame(
	{
		"feature1": [0.3, 0.2, 0.4, 0.1, 0.2, 0.3, 0.5],
		"feature2": [32, 50, 70, 65, 0, 10, 15],
		"label": [1, 0, 1, 0, 1, 0, 0],
	}
)
ml_dataset

feature1,feature2,label
f64,i64,i64
0.3,32,1
0.2,50,0
0.4,70,1
0.1,65,0
0.2,0,1
0.3,10,0
0.5,15,0


In [7]:
ml_dataset.select(
    "label",
    cs.starts_with("feature").map_batches(
        lambda x: softmax(x.to_numpy())
    )
)

label,feature1,feature2
i64,f64,f64
1,0.143782,3.1181e-17
0,0.130099,2.0474e-9
1,0.158904,0.993307
0,0.117719,0.006693
1,0.130099,3.9488e-31
0,0.143782,8.6979e-27
0,0.175616,1.2909e-24


## Applying a Function to Groups

In [8]:
from sklearn.preprocessing import StandardScaler


def scale_temperature(group):
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(group[["temperature"]].to_numpy())
    return group.with_columns(
        pl.Series(values=scaled_values.flatten(), name="scaled_feature")
    )


temperatures = pl.DataFrame(
    {
        "country": ["USA", "USA", "USA", "USA", "NL", "NL", "NL"],
        "temperature": [32, 50, 70, 65, 0, 10, 15],
    }
)

temperatures.group_by("country").map_groups(scale_temperature)

country,temperature,scaled_feature
str,i64,f64
"""USA""",32,-1.502872
"""USA""",50,-0.287066
"""USA""",70,1.063831
"""USA""",65,0.726107
"""NL""",0,-1.336306
"""NL""",10,0.267261
"""NL""",15,1.069045


In [9]:
temperatures = pl.DataFrame(
    {
        "country": ["USA", "USA", "USA", "USA", "NL", "NL", "NL"],
        "temperature": [32, 50, 70, 65, 0, 10, 15],
    }
)

for group, df in temperatures.group_by("country"):
    print(f"{group[0]}:\n{df}\n")

NL:
shape: (3, 2)
┌─────────┬─────────────┐
│ country ┆ temperature │
│ ---     ┆ ---         │
│ str     ┆ i64         │
╞═════════╪═════════════╡
│ NL      ┆ 0           │
│ NL      ┆ 10          │
│ NL      ┆ 15          │
└─────────┴─────────────┘

USA:
shape: (4, 2)
┌─────────┬─────────────┐
│ country ┆ temperature │
│ ---     ┆ ---         │
│ str     ┆ i64         │
╞═════════╪═════════════╡
│ USA     ┆ 32          │
│ USA     ┆ 50          │
│ USA     ┆ 70          │
│ USA     ┆ 65          │
└─────────┴─────────────┘



In [10]:
from functools import lru_cache

from textblob import TextBlob


@lru_cache(maxsize=256)
def analyze_sentiment(review):
    return TextBlob(review).sentiment.polarity


reviews = pl.DataFrame(
    {
        "reviews": [
            "This product is great!",
            "Terrible service.",
            "Okay, but not what I expected.",
            "Excellent! I love it.",
        ]
    }
)

reviews.with_columns(
    sentiment_score=pl.col("reviews").map_elements(
        analyze_sentiment, return_dtype=pl.Float64
    )
)

reviews,sentiment_score
str,f64
"""This product is great!""",1.0
"""Terrible service.""",-1.0
"""Okay, but not what I expected.""",0.2
"""Excellent! I love it.""",0.75


### Applying a Function to an Expression

In [11]:
addresses = pl.DataFrame(
    {
        "address": [
            "Nieuwezijds Voorburgwal 147",
            "Museumstraat 1",
            "Oosterdok 2",
        ]
    }
)


def extract_house_number(input_expr: pl.Expr) -> pl.Expr:
    """Extract the house number from an address String"""
    return input_expr.str.extract(r"\d+", 0).cast(pl.Int64)


addresses.with_columns(
    house_numbers=pl.col("address").pipe(extract_house_number)
)

address,house_numbers
str,i64
"""Nieuwezijds Voorburgwal 147""",147
"""Museumstraat 1""",1
"""Oosterdok 2""",2


### Applying a Function to a DataFrame or LazyFrame

In [12]:
small_numbers = pl.DataFrame({"ints": [2, 4, 6], "floats": [10.0, 20.0, 30.0]})


def scale_the_input(
    df: pl.DataFrame | pl.LazyFrame, scale_factor: int
) -> pl.DataFrame | pl.LazyFrame:
    """Scales the input by the input factor"""
    return df * scale_factor


small_numbers.pipe(scale_the_input, 5)

ints,floats
f64,f64
10.0,50.0
20.0,100.0
30.0,150.0




Polars permite aplicar funciones personalizadas de Python a tus datos mediante varias herramientas expresivas y flexibles. Aquí se resumen los conceptos principales:

- **Expr.map_elements()**  
    Aplica una función a cada elemento de una columna. Útil para operaciones simples y element-wise.

- **Expr.map_batches()**  
    Permite aplicar una función a bloques completos de datos (batches), lo que puede ser útil para operaciones vectorizadas o que requieren contexto de varios elementos.

- **Expr.map_groups()**  
    Ejecuta una función sobre cada grupo en operaciones de agrupamiento (`group_by`). Ideal para cálculos personalizados por grupo.

- **Expr.pipe() y df.pipe()**  
    Permiten encadenar transformaciones aplicando funciones a expresiones o a todo el DataFrame/LazyFrame, facilitando la composición de operaciones complejas.

### Consideraciones de Rendimiento

- Las funciones definidas por el usuario (UDFs) ofrecen gran flexibilidad, pero suelen ser más lentas que las expresiones nativas de Polars, ya que requieren pasar datos entre Rust y Python.
- Si necesitas aplicar funciones de Python repetidamente sobre los mismos datos, puedes usar el decorador `@lru_cache` para cachear resultados y evitar cálculos redundantes.

### Buenas Prácticas

- Prioriza siempre las expresiones nativas de Polars para un mejor rendimiento.
- Usa UDFs solo cuando sea estrictamente necesario.
- Aprovecha herramientas como `@lru_cache` para optimizar funciones costosas y repetitivas.

Al comprender y utilizar estas herramientas, puedes adaptar tus transformaciones de datos en Polars a necesidades específicas, manteniendo la claridad y el rendimiento en tus notebooks.

##  Registering Your Own Namespace

Puedes crear tus propios namespaces personalizados en Polars para extender la API y hacerla más intuitiva. Esto se logra decorando una clase Python con el decorador correspondiente según el contexto (Expr, DataFrame, LazyFrame o Series). Así, puedes agrupar funciones personalizadas bajo un mismo espacio de nombres.

**Tabla 17-2. Decoradores para registrar namespaces personalizados**

| Contexto    | Decorador                                         | Ejemplo de uso                        |
|-------------|---------------------------------------------------|---------------------------------------|
| Expression  | `@pl.api.register_expr_namespace("nombre")`       | `pl.col("col").nombre.func()`         |
| DataFrame   | `@pl.api.register_dataframe_namespace("nombre")`  | `df.nombre.func()`                    |
| LazyFrame   | `@pl.api.register_lazyframe_namespace("nombre")`  | `lf.nombre.func()`                    |
| Series      | `@pl.api.register_series_namespace("nombre")`     | `col.nombre.func()`                   |

**Resumen:**  
1. Define una clase con los métodos que quieras exponer.
2. Decórala con el decorador adecuado y el nombre del namespace.
3. Accede a tus funciones personalizadas desde el DataFrame, Series, Expr o LazyFrame usando el nuevo namespace.

Esto permite crear APIs más limpias y reutilizables para tus flujos de trabajo en Polars.

In [13]:
# Registramos un nuevo namespace "celsius" para expresiones (Expr) en Polars
@pl.api.register_expr_namespace("celsius")  
class Celsius:
    def __init__(self, expr: pl.Expr):  
        # Guardamos la expresión de Polars para operar sobre ella
        self._expr = expr

    def to_fahrenheit(self) -> pl.Expr:  
        # Convierte grados Celsius a Fahrenheit
        return (self._expr * 9 / 5) + 32

    def to_kelvin(self) -> pl.Expr:
        # Convierte grados Celsius a Kelvin
        return self._expr + 273.15

In [14]:
temperatures = pl.DataFrame({"celsius": [0, 10, 20, 30, 40]})
temperatures

celsius
i64
0
10
20
30
40


In [15]:
temperatures.with_columns(
    fahrenheit=pl.col("celsius").celsius.to_fahrenheit(),
    kelvin=pl.col("celsius").celsius.to_kelvin(),
)

celsius,fahrenheit,kelvin
i64,f64,f64
0,32.0,273.15
10,50.0,283.15
20,68.0,293.15
30,86.0,303.15
40,104.0,313.15


## Polars Plugins in Rust

Los plugins de Polars escritos en Rust te permiten extender Polars con funciones de alto rendimiento que se ejecutan directamente en el motor de Polars. Esto ofrece un rendimiento mucho mejor que las funciones Python UDF.

Para crear un plugin:
1. Crea una estructura de proyecto con Rust usando `cargo` y `maturin`
2. Define las funciones en Rust usando el macro `#[polars_expr]`
3. Compila el plugin con `maturin develop --release`
4. Importa y usa el plugin desde Python

El plugin debe compilarse antes de poder usarlo. Descomenta y ejecuta la celda siguiente para compilar:
```python
! cd plugins/hello_world_plugin && uv run maturin develop --release
```

In [16]:
! rustc --version

rustc 1.89.0 (29483883e 2025-08-04)


In [17]:
# Para compilar e instalar el plugin en el entorno actual del notebook:
# Opción 1: Usando el Python del entorno actual
! cd plugins/hello_world_plugin && python -m maturin develop --release

# Opción 2: Si la Opción 1 no funciona, construye el wheel e instala manualmente:
# ! cd plugins/hello_world_plugin && maturin build --release
# ! pip install --force-reinstall plugins/hello_world_plugin/target/wheels/hello_world_plugin-*.whl

✏️ Setting installed package as editable


🍹 Building a mixed python/rust project
🔗 Found pyo3 bindings
🐍 Found CPython 3.12 at C:\Users\sergi\Documents\polars\python-polars-the-definitive-guide\.venv\Scripts\python.exe
    Finished `release` profile [optimized] target(s) in 30.27s
📦 Built wheel for CPython 3.12 to C:\Users\sergi\AppData\Local\Temp\.tmpWh7V2A\hello_world_plugin-1.0.0-cp312-cp312-win_amd64.whl
⚠️ Warning: failed to set package as editable: failed to get version of install backend
🛠 Installed hello_world_plugin-1.0.0


### ¿Cómo instalar el plugin en tu entorno?

Cuando ejecutas `maturin develop`, el plugin se instala en el entorno virtual activo. Para asegurarte de que se instala en el entorno correcto:

**Método 1 (Recomendado - desde el notebook):**
- Ejecuta la celda siguiente que usa `python -m maturin develop --release`
- El comando `python` usará el intérprete de Python del kernel actual del notebook
- Esto garantiza que el plugin se instale exactamente donde lo necesitas

**Método 2 (Desde terminal):**
Si prefieres usar la terminal, activa primero tu entorno:
```powershell
# Navega al directorio del plugin
cd Chapter_17\plugins\hello_world_plugin

# Activa tu entorno (ajusta la ruta según tu entorno)
& 'ruta\a\tu\venv\Scripts\Activate.ps1'

# Compila e instala
maturin develop --release
```

**Verificación:**
Después de la instalación, verifica que funciona ejecutando:
```python
import hello_world_func
print(hello_world_func.__file__)
```

## Estructura del proyecto del plugin

Según el libro "Polars: The Definitive Guide", un plugin sigue esta estructura:

```
/
├── src
│   ├── expressions.rs  # Código Rust de la expresión personalizada
│   └── lib.rs          # Define el módulo Python
├── hello_world_func    # Paquete Python (nombre debe coincidir con [lib] name en Cargo.toml)
│   └── __init__.py     # Registra la función con Polars
└── Cargo.toml          # Metadata del proyecto Rust
```

**Importante:** El nombre del paquete Python (`hello_world_func`) debe coincidir con el `name` en la sección `[lib]` del `Cargo.toml`, no con el nombre del proyecto.

Por eso el import correcto es:
```python
from hello_world_func import hello_world
```

In [18]:
import polars as pl
from hello_world_func import hello_world  
import time

lots_of_strings = pl.DataFrame(
    {
        "a": ["1", "2", "3", "4"] * 1_000_000,
    }
)

times = []
for i in range(10):
    t0 = time.time()
    out = lots_of_strings.with_columns(
        pl.col("a").str.replace_all(r".*", "Hello, world!")
    )
    t1 = time.time()
    times.append(t1 - t0)
print(
    f"Polars native string replace:        {sum(times) / len(times):.5f}"
)  


times = []
for i in range(10):
    t0 = time.time()
    out = lots_of_strings.with_columns(hello_world("a"))  
    t1 = time.time()
    times.append(t1 - t0)
print(f"Our custom made Hello world replace: {sum(times) / len(times):.5f}")

Polars native string replace:        2.78177
Our custom made Hello world replace: 0.49953


## Resumen: Argumentos y Opciones al Registrar Plugins en Polars

Al crear plugins personalizados en Polars (usualmente en Rust), puedes controlar cómo se comportan y optimizan tus funciones mediante argumentos y banderas especiales. Aquí tienes un resumen de los aspectos clave:

### 1. Argumentos Posicionales y por Palabra Clave

- **Múltiples argumentos**: Puedes pasar varias Series como argumentos usando la lista `args` en `register_plugin_function()`.
    ```python
    def args_func(arg1: IntoExpr, arg2: IntoExpr) -> pl.Expr:
        return register_plugin_function(
            plugin_path=PLUGIN_PATH,
            function_name="args_func",
            args=[arg1, arg2],
        )
    ```
- **Keyword arguments (kwargs)**: Puedes pasar argumentos adicionales como diccionario en `kwargs`.
    ```python
    def kwargs_func(expr: IntoExpr, float_arg: float, integer_arg: int, string_arg: str, boolean_arg: bool) -> pl.Expr:
        return register_plugin_function(
            plugin_path=PLUGIN_PATH,
            function_name="kwargs_func",
            args=expr,
            kwargs={
                "float_arg": float_arg,
                "integer_arg": integer_arg,
                "string_arg": string_arg,
                "boolean_arg": boolean_arg,
            },
        )
    ```
- **En Rust**: Define una struct que refleje los kwargs esperados y deriva `Deserialize` de `serde` para deserializarlos.
    ```rust
    #[derive(Deserialize)]
    pub struct MyKwargs {
        float_arg: f64,
        integer_arg: i64,
        string_arg: String,
        boolean_arg: bool,
    }
    ```

### 2. Banderas de Optimización y Comportamiento

- **is_elementwise**:  
  Indica si la función es elemento a elemento. Permite paralelismo y optimizaciones.  
  - `True`: La función se aplica a cada elemento individualmente.
  - ¡Marca correctamente! Si no, los resultados pueden ser incorrectos en operaciones como `group_by` o ventanas.

- **changes_length**:  
  Indica si la función cambia la longitud de la Serie (por ejemplo, `unique()`, `filter()`, `explode()`).  
  - Por defecto es `False`.

- **returns_scalar**:  
  Si la función retorna una lista con un solo elemento, esta bandera permite desempaquetar el valor y devolverlo como escalar.

- **cast_to_supertype**:  
  Si la función acepta argumentos de tipos mixtos, esta bandera fuerza a que todos se casteen al supertipo común antes de ejecutar la función.

- **input_wildcard_expansion**:  
  Si la expresión de entrada es un wildcard (ej. `pl.col("*")`), esta bandera expande la expresión a una lista de Series.

- **pass_name_to_apply**:  
  Si es `True`, la Serie pasada a la función en operaciones `group_by` tendrá su nombre asignado. Útil si el plugin necesita el nombre de la Serie.

---

**Nota:**  
Estas opciones permiten que tus plugins sean más eficientes, seguros y flexibles. Consulta la documentación oficial de Polars y Rust para detalles de implementación y ejemplos avanzados.

In [2]:
import polars as pl
points_and_polygons = pl.DataFrame(
    {
        "point": [[5.0, 5.0], [20.0, 20.0], [20.0, 20.0]],
        "polygon": [
            [[0.0, 0.0], [10.0, 0.0], [10.0, 10.0], [0.0, 10.0]],
            [
                [0.0, 0.0],
                [10.0, 0.0],
                [10.0, 10.0],
            ],
            [[0.0, None], [10.0, 0.0], [10.0, 10.0], [0.0, 10.0], [0.0, 0.0]],
        ],
    }
)
from plugins.polars_geo import polars_geo

# Apply the point_in_polygon function
points_and_polygons.with_columns(
    pl.col("point").geo.point_in_polygon(pl.col("polygon")).alias("in_polygon")
)

point,polygon,in_polygon
list[f64],list[list[f64]],bool
"[5.0, 5.0]","[[0.0, 0.0], [10.0, 0.0], … [0.0, 10.0]]",true
"[20.0, 20.0]","[[0.0, 0.0], [10.0, 0.0], [10.0, 10.0]]",false
"[20.0, 20.0]","[[0.0, null], [10.0, 0.0], … [0.0, 0.0]]",null


## ✅ Verificación del Plugin Geo

El plugin `polars_geo` ha sido implementado exitosamente siguiendo las especificaciones del libro "Python Polars: The Definitive Guide".

### Componentes Implementados:

1. **Rust Code** (`src/expression.rs`):
   - `extract_point()`: Extrae un punto de una Serie
   - `extract_polygon()`: Extrae un polígono de una Serie
   - `geo_point_in_polygon()`: Verifica si un punto está dentro de un polígono
   - `point_in_polygon()`: Función exportada a Python
   - `geo_haversine_distance()`: Calcula la distancia Haversine entre dos puntos
   - `haversine_distance()`: Función exportada a Python

2. **Rust Module** (`src/lib.rs`):
   - Inicializa el módulo PyO3 con PolarsAllocator

3. **Python Registration** (`polars_geo/__init__.py`):
   - Registra las funciones con `register_plugin_function`
   - Define el namespace personalizado `geo` con `@pl.api.register_expr_namespace`
   - Permite usar `.geo.point_in_polygon()` y `.geo.haversine_distance()`

4. **Cargo Configuration** (`Cargo.toml`):
   - Paquete: `polars_geo` v1.0.0
   - Dependencias: `geo`, `polars`, `pyo3`, `pyo3-polars`
   - Crate type: `cdylib`

### Uso del Plugin:

```python
import polars as pl
import polars_geo

# Con namespace personalizado
df.select(
    pl.col("point").geo.point_in_polygon(pl.col("polygon")).alias("is_inside"),
    pl.col("point").geo.haversine_distance(pl.col("to_point")).alias("distance")
)

# O con funciones directas
df.select(
    polars_geo.point_in_polygon(pl.col("point"), pl.col("polygon")),
    polars_geo.haversine_distance(pl.col("point"), pl.col("to_point"))
)
```

### Resultados de las Pruebas:

✅ Compilación exitosa sin warnings
✅ `point_in_polygon` funciona correctamente (detecta puntos dentro/fuera de polígonos)
✅ `haversine_distance` calcula distancias en metros correctamente
✅ Namespace personalizado `.geo` funciona
✅ Funciones directas funcionan

In [3]:
# Importar el plugin geo
import polars_geo

# Crear datos de prueba con puntos y polígonos
test_df = pl.DataFrame({
    "location_name": ["Centro", "Esquina", "Afuera"],
    "point": [[1.0, 1.0], [0.0, 0.0], [3.0, 3.0]],
    "park_boundary": [
        [[0.0, 0.0], [2.0, 0.0], [2.0, 2.0], [0.0, 2.0], [0.0, 0.0]],
        [[0.0, 0.0], [2.0, 0.0], [2.0, 2.0], [0.0, 2.0], [0.0, 0.0]],
        [[0.0, 0.0], [2.0, 0.0], [2.0, 2.0], [0.0, 2.0], [0.0, 0.0]],
    ],
    "destination": [[2.0, 2.0], [1.0, 1.0], [0.0, 0.0]],
})

print("🌍 Prueba 1: Verificar si los puntos están dentro del parque")
result1 = test_df.select(
    pl.col("location_name"),
    pl.col("point").geo.point_in_polygon(pl.col("park_boundary")).alias("inside_park")
)
print(result1)

print("\n📏 Prueba 2: Calcular distancia Haversine al destino (en metros)")
result2 = test_df.select(
    pl.col("location_name"),
    pl.col("point"),
    pl.col("destination"),
    pl.col("point").geo.haversine_distance(pl.col("destination")).alias("distance_m")
)
print(result2)

🌍 Prueba 1: Verificar si los puntos están dentro del parque
shape: (3, 2)
┌───────────────┬─────────────┐
│ location_name ┆ inside_park │
│ ---           ┆ ---         │
│ str           ┆ bool        │
╞═══════════════╪═════════════╡
│ Centro        ┆ true        │
│ Esquina       ┆ false       │
│ Afuera        ┆ false       │
└───────────────┴─────────────┘

📏 Prueba 2: Calcular distancia Haversine al destino (en metros)
shape: (3, 4)
┌───────────────┬────────────┬─────────────┬───────────────┐
│ location_name ┆ point      ┆ destination ┆ distance_m    │
│ ---           ┆ ---        ┆ ---         ┆ ---           │
│ str           ┆ list[f64]  ┆ list[f64]   ┆ f64           │
╞═══════════════╪════════════╪═════════════╪═══════════════╡
│ Centro        ┆ [1.0, 1.0] ┆ [2.0, 2.0]  ┆ 157225.649207 │
│ Esquina       ┆ [0.0, 0.0] ┆ [1.0, 1.0]  ┆ 157249.598474 │
│ Afuera        ┆ [3.0, 3.0] ┆ [0.0, 0.0]  ┆ 471652.939973 │
└───────────────┴────────────┴─────────────┴───────────────┘


C:\Users\sergi\Documents\polars\Chapter_17\plugins\polars_geo\polars_geo\__init__.py:24: UserWarning: Overriding existing custom namespace 'geo' (on 'Expr')
  @pl.api.register_expr_namespace("geo")
